# PINN — Complete End-to-End Implementation

This notebook is the canonical executable implementation for the repository.

It develops a Physics-Informed Neural Network (PINN) for the one-dimensional transient heat equation from the mathematical problem statement through model construction, automatic differentiation, collocation sampling, optimization, validation, visualization, and advanced extensions.


## 1. Mathematical problem

We solve $u_t = \alpha u_{xx}$ for $x\in[-1,1]$ and $t\in[0,1]$, with $u(x,0)=\sin(\pi x)$ and $u(-1,t)=u(1,t)=0$.

For constant $\alpha>0$, the analytical solution is $u(x,t)=e^{-\alpha\pi^2t}\sin(\pi x)$. The analytical solution is used only for validation.


In [ ]:
import math
import random
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

print('PyTorch:', torch.__version__)
print('NumPy:', np.__version__)


In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    alpha: float = 0.1
    x_min: float = -1.0
    x_max: float = 1.0
    t_min: float = 0.0
    t_max: float = 1.0
    n_interior: int = 2500
    n_initial: int = 500
    n_boundary: int = 500
    hidden_dim: int = 64
    hidden_layers: int = 4
    physics_weight: float = 1.0
    initial_weight: float = 10.0
    boundary_weight: float = 10.0
    adam_lr: float = 1e-3
    adam_epochs: int = 1800
    print_every: int = 300
    lbfgs_max_iter: int = 250
    seed: int = 42
    dtype: torch.dtype = torch.float32

cfg = ExperimentConfig()

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(cfg)
print('Device:', device)


## 2. Analytical reference solution

The exact solution is intentionally separated from training so it cannot leak into the loss.


In [ ]:
def exact_solution(x: torch.Tensor, t: torch.Tensor, alpha: float | torch.Tensor) -> torch.Tensor:
    return torch.exp(-alpha * math.pi**2 * t) * torch.sin(math.pi * x)

x_demo = torch.linspace(cfg.x_min, cfg.x_max, 200, dtype=cfg.dtype)
t_demo = torch.full_like(x_demo, 0.5)
u_demo = exact_solution(x_demo, t_demo, cfg.alpha)

plt.figure(figsize=(8, 4))
plt.plot(x_demo, u_demo)
plt.xlabel('x')
plt.ylabel('u(x, 0.5)')
plt.title('Analytical solution')
plt.grid(True, alpha=0.25)
plt.show()


## 3. PINN architecture

The network represents $u_\theta(x,t)$. Inputs are normalized to $[-1,1]$. Smooth Tanh activations are used because derivatives of the network with respect to the inputs are part of the PDE loss.


In [ ]:
class PINN(nn.Module):
    def __init__(self, x_bounds, t_bounds, hidden_dim=64, hidden_layers=4):
        super().__init__()
        if hidden_layers < 1:
            raise ValueError('hidden_layers must be >= 1')
        if x_bounds[0] >= x_bounds[1] or t_bounds[0] >= t_bounds[1]:
            raise ValueError('bounds must be strictly increasing')
        self.register_buffer('lower', torch.tensor([x_bounds[0], t_bounds[0]], dtype=cfg.dtype))
        self.register_buffer('upper', torch.tensor([x_bounds[1], t_bounds[1]], dtype=cfg.dtype))
        layers = [nn.Linear(2, hidden_dim), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), nn.Tanh()])
        layers.append(nn.Linear(hidden_dim, 1))
        self.network = nn.Sequential(*layers)
        self.apply(self._initialize)

    @staticmethod
    def _initialize(module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_normal_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, xt):
        if xt.ndim != 2 or xt.shape[1] != 2:
            raise ValueError(f'Expected shape [N, 2], got {tuple(xt.shape)}')
        normalized = 2.0 * (xt - self.lower) / (self.upper - self.lower) - 1.0
        return self.network(normalized)

seed_everything(cfg.seed)
model = PINN((cfg.x_min, cfg.x_max), (cfg.t_min, cfg.t_max), cfg.hidden_dim, cfg.hidden_layers).to(device=device, dtype=cfg.dtype)
print(model)


## 4. Collocation-point generation

Interior points enforce the PDE. Initial points enforce $u(x,0)$. Boundary points enforce the two Dirichlet boundaries. A local CPU generator makes sampling deterministic without changing the global random state.


In [ ]:
@dataclass(frozen=True)
class CollocationPoints:
    interior: torch.Tensor
    initial: torch.Tensor
    left_boundary: torch.Tensor
    right_boundary: torch.Tensor

    @property
    def boundary(self):
        return torch.cat([self.left_boundary, self.right_boundary], dim=0)

def sample_uniform(n, lower, upper, generator):
    values = torch.rand((n, 2), generator=generator, dtype=cfg.dtype)
    low = torch.tensor(lower, dtype=cfg.dtype)
    high = torch.tensor(upper, dtype=cfg.dtype)
    return low + (high - low) * values

def sample_points(n_interior, n_initial, n_boundary, seed):
    g = torch.Generator(device='cpu')
    g.manual_seed(seed)
    interior = sample_uniform(n_interior, (cfg.x_min, cfg.t_min), (cfg.x_max, cfg.t_max), g)
    x0 = cfg.x_min + (cfg.x_max - cfg.x_min) * torch.rand((n_initial, 1), generator=g, dtype=cfg.dtype)
    initial = torch.cat([x0, torch.full_like(x0, cfg.t_min)], dim=1)
    tl = cfg.t_min + (cfg.t_max - cfg.t_min) * torch.rand((n_boundary, 1), generator=g, dtype=cfg.dtype)
    tr = cfg.t_min + (cfg.t_max - cfg.t_min) * torch.rand((n_boundary, 1), generator=g, dtype=cfg.dtype)
    left = torch.cat([torch.full_like(tl, cfg.x_min), tl], dim=1)
    right = torch.cat([torch.full_like(tr, cfg.x_max), tr], dim=1)
    return CollocationPoints(interior, initial, left, right)

points = sample_points(cfg.n_interior, cfg.n_initial, cfg.n_boundary, cfg.seed)
for name, value in points.__dict__.items():
    print(name, value.shape, value.dtype)


## 5. Automatic differentiation and PDE residual

The residual is $r_\theta=u_t-\alpha u_{xx}$. PyTorch's nested `autograd.grad` calls compute the first and second derivatives exactly through the neural-network computation graph. `create_graph=True` is required so the residual can itself be differentiated during backpropagation.


In [ ]:
def heat_residual(model, xt, alpha):
    if xt.ndim != 2 or xt.shape[1] != 2:
        raise ValueError(f'Expected xt with shape [N, 2], got {tuple(xt.shape)}')
    if not xt.requires_grad:
        xt = xt.clone().detach().requires_grad_(True)
    u = model(xt)
    du = torch.autograd.grad(u, xt, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]
    u_t = du[:, 1:2]
    d2u = torch.autograd.grad(du[:, 0:1], xt, grad_outputs=torch.ones_like(du[:, 0:1]), create_graph=True, retain_graph=True)[0]
    u_xx = d2u[:, 0:1]
    return u_t - alpha * u_xx

xt_check = points.interior[:32].clone().detach().requires_grad_(True)
r_check = heat_residual(model, xt_check, cfg.alpha)
print('shape:', r_check.shape)
print('requires_grad:', r_check.requires_grad)
print('finite:', torch.isfinite(r_check).all().item())


## 6. Composite PINN loss

We minimize a weighted sum of PDE, initial-condition, and boundary-condition mean-square errors.


In [ ]:
def loss_components(model, points, alpha):
    interior = points.interior.to(device).clone().detach().requires_grad_(True)
    physics = heat_residual(model, interior, alpha).square().mean()
    initial = points.initial.to(device)
    initial_target = torch.sin(math.pi * initial[:, 0:1])
    initial_loss = (model(initial) - initial_target).square().mean()
    left = model(points.left_boundary.to(device))
    right = model(points.right_boundary.to(device))
    boundary_loss = 0.5 * (left.square().mean() + right.square().mean())
    total = cfg.physics_weight * physics + cfg.initial_weight * initial_loss + cfg.boundary_weight * boundary_loss
    return total, physics, initial_loss, boundary_loss

total, physics, initial_loss, boundary_loss = loss_components(model, points, cfg.alpha)
print(float(total.detach()), float(physics.detach()), float(initial_loss.detach()), float(boundary_loss.detach()))


## 7. Gradient sanity check

A pure derivative-based loss does not guarantee a gradient for every individual parameter. The correct invariant is that the loss participates in the graph and at least some model parameters receive finite, nonzero gradients.


In [ ]:
model.zero_grad(set_to_none=True)
total, *_ = loss_components(model, points, cfg.alpha)
total.backward()
gradients = [p.grad for p in model.parameters()]
print('Any gradient:', any(g is not None for g in gradients))
print('Any nonzero gradient:', any(g is not None and torch.any(torch.abs(g) > 0) for g in gradients))
print('All existing gradients finite:', all(g is None or torch.isfinite(g).all() for g in gradients))


## 8. Adam training

The baseline uses full-batch collocation points for transparency. Each epoch rebuilds the derivative graph, computes the weighted loss, backpropagates, and updates the network.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.adam_lr)
history = {'total': [], 'physics': [], 'initial': [], 'boundary': []}

for epoch in range(1, cfg.adam_epochs + 1):
    optimizer.zero_grad(set_to_none=True)
    total, physics, initial_loss, boundary_loss = loss_components(model, points, cfg.alpha)
    total.backward()
    optimizer.step()
    history['total'].append(float(total.detach()))
    history['physics'].append(float(physics.detach()))
    history['initial'].append(float(initial_loss.detach()))
    history['boundary'].append(float(boundary_loss.detach()))
    if epoch == 1 or epoch % cfg.print_every == 0 or epoch == cfg.adam_epochs:
        print(f'epoch={epoch:5d} | total={history["total"][-1]:.3e} | physics={history["physics"][-1]:.3e} | initial={history["initial"][-1]:.3e} | boundary={history["boundary"][-1]:.3e}')


In [ ]:
plt.figure(figsize=(9, 5))
plt.semilogy(history['total'], label='total')
plt.semilogy(history['physics'], label='physics')
plt.semilogy(history['initial'], label='initial')
plt.semilogy(history['boundary'], label='boundary')
plt.xlabel('Adam epoch')
plt.ylabel('Loss')
plt.title('PINN training history')
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()


## 9. Optional L-BFGS refinement

A common PINN optimization schedule is Adam followed by L-BFGS. The closure must recompute both the loss and gradients whenever L-BFGS requests them.


In [ ]:
lbfgs = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=cfg.lbfgs_max_iter, history_size=50, tolerance_grad=1e-7, tolerance_change=1e-9, line_search_fn='strong_wolfe')

def closure():
    lbfgs.zero_grad(set_to_none=True)
    total, *_ = loss_components(model, points, cfg.alpha)
    total.backward()
    return total

before = float(loss_components(model, points, cfg.alpha)[0].detach())
with torch.enable_grad():
    lbfgs.step(closure)
after = float(loss_components(model, points, cfg.alpha)[0].detach())
print('Loss before L-BFGS:', before)
print('Loss after L-BFGS:', after)


## 10. Quantitative validation

The evaluation grid is separate from the training points. We report RMSE, MAE, $L_\infty$, relative $L_2$, and relative $L_\infty$ errors.


In [ ]:
@torch.no_grad()
def predict(model, xt):
    model.eval()
    return model(xt.to(device))

n_x, n_t = 201, 121
x_eval = torch.linspace(cfg.x_min, cfg.x_max, n_x, dtype=cfg.dtype)
t_eval = torch.linspace(cfg.t_min, cfg.t_max, n_t, dtype=cfg.dtype)
X, T = torch.meshgrid(x_eval, t_eval, indexing='ij')
grid = torch.stack([X.reshape(-1), T.reshape(-1)], dim=1)
prediction = predict(model, grid).reshape(n_x, n_t).cpu()
reference = exact_solution(X, T, cfg.alpha).cpu()
error = prediction - reference
absolute_error = error.abs()
rmse = torch.sqrt(torch.mean(error.square()))
mae = absolute_error.mean()
linf = absolute_error.max()
relative_l2 = torch.linalg.vector_norm(error) / (torch.linalg.vector_norm(reference) + torch.finfo(prediction.dtype).eps)
relative_linf = linf / (reference.abs().max() + torch.finfo(prediction.dtype).eps)
print(f'RMSE: {rmse.item():.6e}')
print(f'MAE: {mae.item():.6e}')
print(f'L_inf: {linf.item():.6e}')
print(f'relative L2: {relative_l2.item():.6e}')
print(f'relative L_inf: {relative_linf.item():.6e}')


## 11. Field and error visualization


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, field, title in zip(axes, [prediction, reference, absolute_error], ['PINN', 'Exact', 'Absolute error']):
    im = ax.imshow(field.numpy(), origin='lower', aspect='auto', extent=[cfg.t_min, cfg.t_max, cfg.x_min, cfg.x_max])
    ax.set_title(title)
    ax.set_xlabel('t')
    ax.set_ylabel('x')
    fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
selected_times = [0.0, 0.25, 0.5, 1.0]
plt.figure(figsize=(9, 5))
for t_value in selected_times:
    xt = torch.stack([x_eval, torch.full_like(x_eval, t_value)], dim=1)
    pred = predict(model, xt).squeeze(-1).cpu()
    ref = exact_solution(x_eval, torch.full_like(x_eval, t_value), cfg.alpha).cpu()
    plt.plot(x_eval, pred, label=f'PINN t={t_value:g}')
    plt.plot(x_eval, ref, '--', label=f'Exact t={t_value:g}')
plt.xlabel('x')
plt.ylabel('u')
plt.title('Spatial profiles')
plt.legend(ncol=2)
plt.grid(True, alpha=0.25)
plt.show()


## 12. Independent PDE and constraint diagnostics

An independent residual test set checks whether the learned field satisfies the mathematical constraints away from the training points.


In [ ]:
test_points = sample_points(1500, 400, 400, cfg.seed + 1)
independent = test_points.interior.to(device).clone().detach().requires_grad_(True)
with torch.enable_grad():
    independent_residual = heat_residual(model, independent, cfg.alpha)
physics_rmse = torch.sqrt(independent_residual.square().mean()).item()
with torch.no_grad():
    init_pred = model(test_points.initial.to(device))
    init_target = torch.sin(math.pi * test_points.initial[:, 0:1]).to(device)
    left_pred = model(test_points.left_boundary.to(device))
    right_pred = model(test_points.right_boundary.to(device))
initial_rmse = torch.sqrt((init_pred - init_target).square().mean()).item()
left_rmse = torch.sqrt(left_pred.square().mean()).item()
right_rmse = torch.sqrt(right_pred.square().mean()).item()
print(f'Independent PDE residual RMSE: {physics_rmse:.6e}')
print(f'Initial RMSE: {initial_rmse:.6e}')
print(f'Left boundary RMSE: {left_rmse:.6e}')
print(f'Right boundary RMSE: {right_rmse:.6e}')


## 13. Analytical residual identity

The exact solution should satisfy $u_t-\alpha u_{xx}=0$. This is a useful independent implementation test.


In [ ]:
xt_exact = torch.rand((256, 2), generator=torch.Generator().manual_seed(cfg.seed + 10), dtype=cfg.dtype)
xt_exact[:, 0] = cfg.x_min + (cfg.x_max - cfg.x_min) * xt_exact[:, 0]
xt_exact[:, 1] = cfg.t_min + (cfg.t_max - cfg.t_min) * xt_exact[:, 1]
xt_exact.requires_grad_(True)
u_exact = exact_solution(xt_exact[:, 0:1], xt_exact[:, 1:2], cfg.alpha)
du = torch.autograd.grad(u_exact, xt_exact, grad_outputs=torch.ones_like(u_exact), create_graph=True)[0]
d2 = torch.autograd.grad(du[:, 0:1], xt_exact, grad_outputs=torch.ones_like(du[:, 0:1]))[0]
analytical_residual = du[:, 1:2] - cfg.alpha * d2[:, 0:1]
print('Analytical residual max abs:', analytical_residual.abs().max().item())


## 14. Reproducibility check

Model initialization must be seeded before model construction. This differs from seeding only after a model has already sampled its parameters.


In [ ]:
def build_reproducible_model(seed):
    seed_everything(seed)
    return PINN((cfg.x_min, cfg.x_max), (cfg.t_min, cfg.t_max), hidden_dim=32, hidden_layers=2).to(device=device, dtype=cfg.dtype)

m1 = build_reproducible_model(123)
m2 = build_reproducible_model(123)
same_initialization = all(torch.equal(a, b) for a, b in zip(m1.parameters(), m2.parameters()))
print('Same initial parameters:', same_initialization)


## 15. Residual-based adaptive refinement

Generate a larger candidate pool, score the PDE residual, and retain high-residual locations as a targeted collocation set.


In [ ]:
candidate_generator = torch.Generator(device='cpu').manual_seed(cfg.seed + 20)
candidate = sample_uniform(8000, (cfg.x_min, cfg.t_min), (cfg.x_max, cfg.t_max), candidate_generator)
candidate_grad = candidate.to(device).clone().detach().requires_grad_(True)
with torch.enable_grad():
    candidate_residual = heat_residual(model, candidate_grad, cfg.alpha)
scores = candidate_residual.detach().abs().squeeze(1)
top_k = 400
top_indices = torch.topk(scores, k=top_k).indices
refined_points = candidate[top_indices.cpu()]
print('Candidate points:', len(candidate))
print('Selected points:', len(refined_points))
print('Median candidate residual:', scores.median().item())
print('Median selected residual:', scores[top_indices].median().item())


## 16. Inverse PINN: estimate unknown diffusivity

Treat $\alpha$ as trainable while enforcing positivity with $\alpha=softplus(\beta)$. Synthetic observations from the analytical solution provide the data term.


In [ ]:
class InversePINN(nn.Module):
    def __init__(self, initial_alpha):
        super().__init__()
        self.field = PINN((cfg.x_min, cfg.x_max), (cfg.t_min, cfg.t_max), hidden_dim=48, hidden_layers=3)
        self.raw_alpha = nn.Parameter(torch.tensor(math.log(math.expm1(initial_alpha)), dtype=cfg.dtype))
    @property
    def alpha(self):
        return torch.nn.functional.softplus(self.raw_alpha)
    def forward(self, xt):
        return self.field(xt)

seed_everything(cfg.seed + 100)
inverse_model = InversePINN(0.2).to(device=device, dtype=cfg.dtype)
obs = sample_uniform(300, (cfg.x_min, cfg.t_min), (cfg.x_max, cfg.t_max), torch.Generator().manual_seed(cfg.seed + 101))
obs_u = exact_solution(obs[:, 0:1], obs[:, 1:2], cfg.alpha)
print('Initial alpha estimate:', inverse_model.alpha.item())


In [ ]:
inverse_optimizer = torch.optim.Adam(inverse_model.parameters(), lr=1e-3)
inverse_history = {'total': [], 'physics': [], 'data': [], 'alpha': []}
for epoch in range(1, 1201):
    inverse_optimizer.zero_grad(set_to_none=True)
    interior = points.interior.to(device).clone().detach().requires_grad_(True)
    residual = heat_residual(inverse_model, interior, inverse_model.alpha)
    physics_loss = residual.square().mean()
    data_prediction = inverse_model(obs.to(device))
    data_loss = (data_prediction - obs_u.to(device)).square().mean()
    total_loss = physics_loss + 20.0 * data_loss
    total_loss.backward()
    inverse_optimizer.step()
    inverse_history['total'].append(float(total_loss.detach()))
    inverse_history['physics'].append(float(physics_loss.detach()))
    inverse_history['data'].append(float(data_loss.detach()))
    inverse_history['alpha'].append(float(inverse_model.alpha.detach()))
    if epoch == 1 or epoch % 300 == 0 or epoch == 1200:
        print(f'epoch={epoch:4d} | total={inverse_history["total"][-1]:.3e} | data={inverse_history["data"][-1]:.3e} | alpha={inverse_history["alpha"][-1]:.6f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(inverse_history['total'], label='total')
axes[0].semilogy(inverse_history['physics'], label='physics')
axes[0].semilogy(inverse_history['data'], label='data')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('loss')
axes[0].legend()
axes[0].grid(True, alpha=0.25)
axes[1].plot(inverse_history['alpha'], label='estimated alpha')
axes[1].axhline(cfg.alpha, linestyle='--', label='true alpha')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('alpha')
axes[1].legend()
axes[1].grid(True, alpha=0.25)
plt.tight_layout()
plt.show()
print('True alpha:', cfg.alpha)
print('Final estimate:', inverse_model.alpha.item())


## 17. Reusable repository API

The notebook implementation above is intentionally explicit. The repository also packages the reusable abstractions as `MLP`, `heat_residual`, `sample_heat_equation`, `PINNConfig`, and `PINNTrainer`.


In [ ]:
try:
    from pinn import MLP, PINNConfig, PINNTrainer, heat_exact_solution, heat_residual as package_heat_residual, sample_heat_equation
    package_available = True
    print('Repository package import: OK')
except ImportError as exc:
    package_available = False
    print('Repository package not installed. From repo root run: python -m pip install -e .')
    print(exc)


In [ ]:
if package_available:
    package_points = sample_heat_equation(1000, 250, 250, x_bounds=(cfg.x_min, cfg.x_max), t_bounds=(cfg.t_min, cfg.t_max), seed=cfg.seed)
    package_model = MLP(input_dim=2, output_dim=1, hidden_dim=48, hidden_layers=3, input_bounds=((cfg.x_min, cfg.x_max), (cfg.t_min, cfg.t_max)))
    package_config = PINNConfig(alpha=cfg.alpha, physics_weight=1.0, initial_weight=10.0, boundary_weight=10.0, learning_rate=1e-3, epochs=600, seed=cfg.seed, log_every=300, use_lbfgs=False)
    package_trainer = PINNTrainer(package_model, package_config)
    package_history = package_trainer.train(package_points)
    print('Repository trainer completed. Final loss:', package_history.total[-1])


In [ ]:
if package_available:
    package_x = torch.linspace(cfg.x_min, cfg.x_max, 101, dtype=cfg.dtype)
    package_t = torch.linspace(cfg.t_min, cfg.t_max, 61, dtype=cfg.dtype)
    PX, PT = torch.meshgrid(package_x, package_t, indexing='ij')
    package_grid = torch.stack([PX.reshape(-1), PT.reshape(-1)], dim=1)
    package_prediction = package_trainer.predict(package_grid)
    package_reference = heat_exact_solution(package_grid, cfg.alpha)
    package_error = package_prediction - package_reference
    print('Package RMSE:', torch.sqrt(package_error.square().mean()).item())
    print('Package L_inf:', package_error.abs().max().item())


## 18. Final structural checks


In [ ]:
assert r_check.shape == (32, 1)
assert r_check.requires_grad
assert torch.isfinite(r_check).all()
assert torch.isfinite(prediction).all()
assert torch.isfinite(reference).all()
assert math.isfinite(rmse.item())
assert math.isfinite(mae.item())
assert math.isfinite(linf.item())
assert math.isfinite(relative_l2.item())
assert analytical_residual.abs().max().item() < 1e-5
print('All core notebook assertions passed.')


## 19. Implementation map

```text
Physical problem
      |
      v
Analytical reference
      |
      v
MLP u_theta(x,t)
      |
      +----------------+
      |                |
      v                v
Autograd          Initial / boundary predictions
u_t, u_xx             |
      |                v
      v           IC / BC losses
PDE residual          |
      +--------+-------+
               v
       weighted total loss
               |
               v
       Adam -> optional L-BFGS
               |
               v
       independent validation
          |      |       |
          v      v       v
        norms   plots  residuals
               |
               v
        adaptive / inverse
           extensions
```

The central idea is that the network is not trained only against observed solution values. Its input derivatives are inserted into the governing differential equation, making the PDE operator part of the optimization objective.
